# PAL observation-noise sweep

This notebook mirrors the existing PNN observation-noise experiment, but replaces the
periodic neural network residual learner by the PAL residual learner used in the baseline
experiments.

The experiment keeps the following points paired with the PNN sweep:

- hidden mapping-class element: `AB`
- baseline residual amplitude: \(\varepsilon=0.05\)
- candidate set: the same 9 mapping-class elements
- observation-noise grid:
  \(\sigma_{\rm noise}\in\{0,0.01,0.02,0.05,0.10,0.20\}\)
- the same input-sampling rule and the same seed convention
- within each seed, the same underlying Gaussian noise draws are reused for every noise level
- every candidate is freshly trained at every noise level; **no warm start** is used
- candidate selection is **unpenalized validation loss** (\(\lambda=0\)), matching the revised manuscript

Two modes are provided:

- `"pilot"`: one full seed (`seed=0`) over all six noise levels
- `"full"`: five full seeds (`0,1,2,3,4`) over all six noise levels

A `"smoke"` mode is also included only for code testing. It is not a scientific run.

In [ ]:
from pathlib import Path
import json
import time
import random
import shutil

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MODE = "full"   # "pilot", "full", or "smoke"

NOISE_LEVELS = [0.0, 0.01, 0.02, 0.05, 0.10, 0.20]

N_TRAIN = 2000
N_VAL   = 1000
N_TEST  = 1000

EPSILON = 0.05

EPOCHS = 1000
LR = 1e-3
PAL_DEGREE = 2
PAL_STEPS = 40
PAL_HIDDEN_SCALE = 0.1

# The revised paper selects by unpenalized validation loss.
PRIMARY_LAMBDA = 0.0

# Optional audit only: this was the tiny reference penalty in the older code.
REFERENCE_LAMBDA = 1e-7

RESUME = True

if MODE == "pilot":
    SEEDS = [0]
    RUN_TAG = "pal_noise_pilot"
elif MODE == "full":
    SEEDS = [0, 1, 2, 3, 4]
    RUN_TAG = "pal_noise_5seeds"
elif MODE == "smoke":
    # Code test only -- do not use these results scientifically.
    SEEDS = [0]
    NOISE_LEVELS = [0.0, 0.05]
    N_TRAIN = 40
    N_VAL = 20
    N_TEST = 20
    EPOCHS = 2
    PAL_STEPS = 2
    RUN_TAG = "pal_noise_smoke"
else:
    raise ValueError("MODE must be 'pilot', 'full', or 'smoke'")

device = torch.device("cpu")

# Colab-friendly, but also works locally.
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "results" / "pal_observation_noise" / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_CSV = OUTPUT_DIR / "candidate_results.csv"
RUN_SUMMARY_CSV = OUTPUT_DIR / "run_summaries.csv"
AGGREGATE_CSV = OUTPUT_DIR / "aggregate_summary.csv"
PAPER_TABLE_CSV = OUTPUT_DIR / "paper_table.csv"
CONFIG_JSON = OUTPUT_DIR / "run_config.json"

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not enforce deterministic algorithms:", e)

config = {
    "mode": MODE,
    "seeds": SEEDS,
    "noise_levels": NOISE_LEVELS,
    "n_train": N_TRAIN,
    "n_val": N_VAL,
    "n_test": N_TEST,
    "epsilon": EPSILON,
    "epochs": EPOCHS,
    "lr": LR,
    "pal_degree": PAL_DEGREE,
    "pal_steps": PAL_STEPS,
    "pal_hidden_scale": PAL_HIDDEN_SCALE,
    "primary_lambda": PRIMARY_LAMBDA,
    "reference_lambda": REFERENCE_LAMBDA,
    "device": str(device),
    "pytorch": torch.__version__,
    "numpy": np.__version__,
}
CONFIG_JSON.write_text(json.dumps(config, indent=2), encoding="utf-8")

print("device:", device)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MODE:", MODE)
print("SEEDS:", SEEDS)
print("NOISE_LEVELS:", NOISE_LEVELS)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# ------------------------------------------------------------
# Torus loss
# ------------------------------------------------------------

def torus_diff(pred, target):
    return (pred - target + 0.5) % 1.0 - 0.5

def torus_mse(pred, target):
    return torch.mean(torus_diff(pred, target) ** 2)

In [ ]:
# ------------------------------------------------------------
# Original 9 candidate mapping-class elements
# ------------------------------------------------------------

A = np.array([[1, 1],
              [0, 1]], dtype=np.int64)

A_INV = np.array([[ 1, -1],
                  [ 0,  1]], dtype=np.int64)

B = np.array([[1, 0],
              [1, 1]], dtype=np.int64)

B_INV = np.array([[ 1,  0],
                  [-1,  1]], dtype=np.int64)

TOKEN_MATRIX = {
    "A": A,
    "Ainv": A_INV,
    "B": B,
    "Binv": B_INV,
}

def word_matrix(tokens):
    M = np.eye(2, dtype=np.int64)
    for token in tokens:
        M = M @ TOKEN_MATRIX[token]
    return M

def matrix_map(M):
    Mt = torch.tensor(M.T, dtype=torch.float32, device=device)

    def G(x):
        return (x @ Mt) % 1.0

    return G

candidate_tokens = {
    "id":   [],
    "A":    ["A"],
    "A^-1": ["Ainv"],
    "B":    ["B"],
    "B^-1": ["Binv"],
    "AB":   ["A", "B"],
    "BA":   ["B", "A"],
    "A^2":  ["A", "A"],
    "B^2":  ["B", "B"],
}

candidates = {}
for name, tokens in candidate_tokens.items():
    M = word_matrix(tokens)
    candidates[name] = {
        "tokens": tokens,
        "matrix": M,
        "length": len(tokens),
        "map": matrix_map(M),
    }

# Verify that the candidate list contains distinct SL(2,Z) elements.
seen = {}
for name, info in candidates.items():
    key = tuple(info["matrix"].reshape(-1).tolist())
    if key in seen:
        raise ValueError(f"Duplicate element: {seen[key]} and {name}")
    seen[key] = name

HIDDEN_WORD = "AB"

display(
    pd.DataFrame(
        [
            {
                "candidate": name,
                "length": info["length"],
                "matrix": str(info["matrix"].tolist()),
            }
            for name, info in candidates.items()
        ]
    )
)

In [ ]:
# ------------------------------------------------------------
# Baseline residual deformation and clean target
# ------------------------------------------------------------

def F0(x, eps=0.05):
    u = (
        x[..., 0]
        + eps * torch.sin(2 * torch.pi * (x[..., 0] - x[..., 1]))
    ) % 1.0

    v = (
        x[..., 1]
        + eps * torch.sin(2 * torch.pi * x[..., 1])
    ) % 1.0

    return torch.stack([u, v], dim=-1)

def clean_target(x):
    return F0(
        candidates[HIDDEN_WORD]["map"](x),
        eps=EPSILON,
    )

def add_wrapped_noise(y_clean, z, sigma):
    return (y_clean + sigma * z) % 1.0

In [ ]:
# ------------------------------------------------------------
# PAL residual learner: same parameterization as the baseline PAL notebook
# ------------------------------------------------------------

class PALResidualTorus(torch.nn.Module):
    def __init__(
        self,
        ks=None,
        D=2,
        hidden_scale=0.1,
        steps=40,
    ):
        super().__init__()

        if ks is None:
            ks = [
                (0, 0),
                (1, 0), (0, 1),
                (1, 1), (1, -1),
                (2, 0), (0, 2),
            ]

        self.register_buffer(
            "ks",
            torch.tensor(ks, dtype=torch.float32),
        )
        self.D = D
        self.steps = steps
        self.hidden_scale = hidden_scale
        self.omega = 2 * torch.pi

        # Monomials s1^p s2^q with p+q <= D.
        self.pq = [
            (p, q)
            for p in range(D + 1)
            for q in range(D + 1 - p)
        ]

        M = len(ks)
        self.num_space_features = 2 * M
        self.num_time_features = len(self.pq)

        self.theta = torch.nn.Parameter(
            0.01 * torch.randn(
                self.num_space_features,
                self.num_time_features,
                2,
            )
        )

    def time_features(self, s):
        s1 = s[:, 0]
        s2 = s[:, 1]

        feats = []
        for p, q in self.pq:
            feats.append((s1 ** p) * (s2 ** q))

        return torch.stack(feats, dim=-1)

    def space_features(self, x):
        phase = 2 * torch.pi * (x @ self.ks.T)
        c = torch.cos(phase)
        s = torch.sin(phase)
        return torch.cat([c, s], dim=-1)

    def vector_field_x(self, x, s):
        phi = self.space_features(x)
        P = self.time_features(s)

        dx = torch.einsum(
            "bp,bq,pqd->bd",
            phi,
            P,
            self.theta,
        )
        return self.hidden_scale * dx

    def rhs(self, z):
        x = z[:, :2]
        s = z[:, 2:]

        dx = self.vector_field_x(x, s)
        ds = self.omega * torch.stack(
            [-s[:, 1], s[:, 0]],
            dim=-1,
        )
        return torch.cat([dx, ds], dim=-1)

    def rk4_step(self, z, dt):
        k1 = self.rhs(z)
        k2 = self.rhs(z + 0.5 * dt * k1)
        k3 = self.rhs(z + 0.5 * dt * k2)
        k4 = self.rhs(z + dt * k3)

        return z + (dt / 6.0) * (
            k1 + 2*k2 + 2*k3 + k4
        )

    def forward(self, x):
        batch = x.shape[0]

        s0 = torch.zeros(batch, 2, device=x.device)
        s0[:, 0] = 1.0

        z = torch.cat([x, s0], dim=-1)
        dt = 1.0 / self.steps

        for _ in range(self.steps):
            z = self.rk4_step(z, dt)

        return z[:, :2] % 1.0

In [ ]:
# ------------------------------------------------------------
# Candidate-wise PAL training
# ------------------------------------------------------------

def train_one_candidate_pal(
    candidate_name,
    gx_train,
    y_train_obs,
    gx_val,
    y_val_obs,
    gx_test,
    y_test_clean,
    y_test_obs,
    seed,
):
    # Same initialization seed for every candidate within a seed/noise run.
    # We also reuse the same initialization across noise levels for a given seed.
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = PALResidualTorus(
        D=PAL_DEGREE,
        steps=PAL_STEPS,
        hidden_scale=PAL_HIDDEN_SCALE,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
    )

    model.train()
    for _ in range(EPOCHS):
        optimizer.zero_grad()

        pred = model(gx_train)
        loss = torus_mse(pred, y_train_obs)

        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_train = model(gx_train)
        pred_val = model(gx_val)
        pred_test = model(gx_test)

        train_loss_noisy = torus_mse(
            pred_train, y_train_obs
        ).item()

        val_loss_noisy = torus_mse(
            pred_val, y_val_obs
        ).item()

        test_loss_clean = torus_mse(
            pred_test, y_test_clean
        ).item()

        test_loss_noisy = torus_mse(
            pred_test, y_test_obs
        ).item()

    word_length = candidates[candidate_name]["length"]

    reference_penalized_score = (
        val_loss_noisy
        + REFERENCE_LAMBDA * word_length
    )

    return {
        "train_loss_noisy": train_loss_noisy,
        "val_loss_noisy": val_loss_noisy,
        "test_loss_clean": test_loss_clean,
        "test_loss_noisy": test_loss_noisy,
        "word_length": word_length,
        "reference_penalized_score": reference_penalized_score,
    }

## Run the sweep

For each seed, input samples and base Gaussian noise draws are generated once.
The same noise draw is then multiplied by each \(\sigma_{\rm noise}\), exactly as
in the PNN noise experiment.

Candidate-transformed inputs \(G(x)\) are cached because they do not depend on
the noise level. Each PAL model is nevertheless trained from a fresh initialization
for every `(seed, noise level, candidate)` combination.

Candidate-level results are autosaved after every training. If the runtime stops and
the output directory survives, rerunning with `RESUME=True` skips completed trainings.

In [ ]:
# ------------------------------------------------------------
# Run PAL noise sweep
# ------------------------------------------------------------

if RESUME and CANDIDATE_CSV.exists():
    candidate_rows = pd.read_csv(CANDIDATE_CSV).to_dict("records")
    print(f"Resuming from {len(candidate_rows)} completed candidate trainings.")
else:
    candidate_rows = []

def completed_keys(rows):
    return {
        (
            int(r["seed"]),
            f"{float(r['sigma_noise']):.6f}",
            str(r["candidate"]),
        )
        for r in rows
    }

done = completed_keys(candidate_rows)

total_trainings = (
    len(SEEDS)
    * len(NOISE_LEVELS)
    * len(candidates)
)

global_start = time.time()
new_trainings = 0

for seed in SEEDS:
    # --------------------------------------------------------
    # Match the PNN sampling/noise convention.
    # --------------------------------------------------------
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    x_train = torch.rand(N_TRAIN, 2, device=device)
    x_val   = torch.rand(N_VAL, 2, device=device)
    x_test  = torch.rand(N_TEST, 2, device=device)

    with torch.no_grad():
        y_train_clean = clean_target(x_train)
        y_val_clean   = clean_target(x_val)
        y_test_clean  = clean_target(x_test)

    # Base noise is drawn once per seed and rescaled for each sigma.
    z_train = torch.randn(N_TRAIN, 2, device=device)
    z_val   = torch.randn(N_VAL, 2, device=device)
    z_test  = torch.randn(N_TEST, 2, device=device)

    # Cache candidate-transformed inputs.
    transformed = {}
    with torch.no_grad():
        for candidate_name, info in candidates.items():
            G = info["map"]
            transformed[candidate_name] = {
                "train": G(x_train),
                "val": G(x_val),
                "test": G(x_test),
            }

    for sigma in NOISE_LEVELS:
        with torch.no_grad():
            y_train_obs = add_wrapped_noise(
                y_train_clean, z_train, sigma
            )
            y_val_obs = add_wrapped_noise(
                y_val_clean, z_val, sigma
            )
            y_test_obs = add_wrapped_noise(
                y_test_clean, z_test, sigma
            )

        print("=" * 78)
        print(
            f"seed={seed} | sigma_noise={sigma:.3f} "
            f"| hidden={HIDDEN_WORD}"
        )
        print("=" * 78)

        for candidate_name in candidates:
            key = (
                int(seed),
                f"{float(sigma):.6f}",
                candidate_name,
            )

            if key in done:
                print(f"{candidate_name:6s} already completed -- skipping")
                continue

            t0 = time.time()
            gx = transformed[candidate_name]

            result = train_one_candidate_pal(
                candidate_name,
                gx["train"],
                y_train_obs,
                gx["val"],
                y_val_obs,
                gx["test"],
                y_test_clean,
                y_test_obs,
                seed=seed,
            )

            row = {
                "seed": seed,
                "sigma_noise": sigma,
                "candidate": candidate_name,
                **result,
            }

            candidate_rows.append(row)
            done.add(key)
            new_trainings += 1

            # Autosave after every candidate.
            pd.DataFrame(candidate_rows).to_csv(
                CANDIDATE_CSV,
                index=False,
            )

            elapsed = time.time() - global_start
            avg = elapsed / max(new_trainings, 1)

            remaining_total = total_trainings - len(done)
            eta_min = avg * max(remaining_total, 0) / 60.0

            print(
                f"{candidate_name:6s} "
                f"val_noisy={result['val_loss_noisy']:.4e} "
                f"clean_test={result['test_loss_clean']:.4e} "
                f"({time.time()-t0:.1f}s; rough ETA ~{eta_min:.1f} min)"
            )

candidate_df = pd.DataFrame(candidate_rows)
candidate_df = candidate_df.sort_values(
    ["sigma_noise", "seed", "val_loss_noisy"]
).reset_index(drop=True)

print("\nPAL noise sweep training stage completed.")
print("candidate rows:", len(candidate_df))
display(candidate_df.head(20))

In [ ]:
# ------------------------------------------------------------
# Build per-run summaries using UNPENALIZED validation selection
# ------------------------------------------------------------

run_summaries = []

for seed in SEEDS:
    for sigma in NOISE_LEVELS:
        d = candidate_df[
            (candidate_df["seed"] == seed)
            & np.isclose(
                candidate_df["sigma_noise"],
                sigma,
            )
        ].copy()

        if len(d) != len(candidates):
            print(
                f"Skipping incomplete condition: "
                f"seed={seed}, sigma={sigma}; "
                f"found {len(d)}/{len(candidates)} candidates."
            )
            continue

        target_row = d[
            d["candidate"] == HIDDEN_WORD
        ].iloc[0]

        wrong_df = d[
            d["candidate"] != HIDDEN_WORD
        ]

        best_wrong_row = wrong_df.loc[
            wrong_df["val_loss_noisy"].idxmin()
        ]

        selected_row = d.loc[
            d["val_loss_noisy"].idxmin()
        ]

        selected_ref_row = d.loc[
            d["reference_penalized_score"].idxmin()
        ]

        validation_margin = (
            best_wrong_row["val_loss_noisy"]
            - target_row["val_loss_noisy"]
        )

        run_summaries.append({
            "seed": seed,
            "sigma_noise": sigma,
            "selected_candidate": selected_row["candidate"],
            "correct_selection": (
                selected_row["candidate"] == HIDDEN_WORD
            ),
            "selected_reference_lambda": (
                selected_ref_row["candidate"]
            ),
            "reference_lambda_agrees": (
                selected_ref_row["candidate"]
                == selected_row["candidate"]
            ),
            "target_val_loss_noisy": (
                target_row["val_loss_noisy"]
            ),
            "target_clean_test_loss": (
                target_row["test_loss_clean"]
            ),
            "best_wrong_candidate": (
                best_wrong_row["candidate"]
            ),
            "best_wrong_val_loss_noisy": (
                best_wrong_row["val_loss_noisy"]
            ),
            "best_wrong_clean_test_loss": (
                best_wrong_row["test_loss_clean"]
            ),
            "validation_margin": validation_margin,
            "selected_clean_test_loss": (
                selected_row["test_loss_clean"]
            ),
            "selected_noisy_test_loss": (
                selected_row["test_loss_noisy"]
            ),
        })

run_summary_df = pd.DataFrame(run_summaries).sort_values(
    ["sigma_noise", "seed"]
).reset_index(drop=True)

run_summary_df.to_csv(
    RUN_SUMMARY_CSV,
    index=False,
)

display(run_summary_df)

print("\nUnpenalized selection counts:")
display(
    pd.crosstab(
        run_summary_df["sigma_noise"],
        run_summary_df["selected_candidate"],
    )
)

if len(run_summary_df):
    print(
        "\nReference lambda agrees with unpenalized selector in "
        f"{int(run_summary_df['reference_lambda_agrees'].sum())}/"
        f"{len(run_summary_df)} completed conditions."
    )

In [ ]:
# ------------------------------------------------------------
# Aggregate over seeds
# ------------------------------------------------------------

def sample_sd(x):
    x = np.asarray(x, dtype=float)
    return (
        np.std(x, ddof=1)
        if len(x) > 1
        else np.nan
    )

summary_rows = []

for sigma in NOISE_LEVELS:
    d = run_summary_df[
        np.isclose(
            run_summary_df["sigma_noise"],
            sigma,
        )
    ]

    if len(d) == 0:
        continue

    summary_rows.append({
        "sigma_noise": sigma,
        "correct_count": int(
            d["correct_selection"].sum()
        ),
        "n_seeds": len(d),
        "selection_rate": float(
            d["correct_selection"].mean()
        ),
        "target_val_noisy_mean": float(
            d["target_val_loss_noisy"].mean()
        ),
        "target_val_noisy_sd": sample_sd(
            d["target_val_loss_noisy"]
        ),
        "margin_mean": float(
            d["validation_margin"].mean()
        ),
        "margin_sd": sample_sd(
            d["validation_margin"]
        ),
        "target_clean_test_mean": float(
            d["target_clean_test_loss"].mean()
        ),
        "target_clean_test_sd": sample_sd(
            d["target_clean_test_loss"]
        ),
        "selected_clean_test_mean": float(
            d["selected_clean_test_loss"].mean()
        ),
        "selected_clean_test_sd": sample_sd(
            d["selected_clean_test_loss"]
        ),
        "selected_noisy_test_mean": float(
            d["selected_noisy_test_loss"].mean()
        ),
        "selected_noisy_test_sd": sample_sd(
            d["selected_noisy_test_loss"]
        ),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    AGGREGATE_CSV,
    index=False,
)

display(summary_df)

paper_rows = []
for _, r in summary_df.iterrows():
    n = int(r["n_seeds"])

    if n > 1:
        val_text = (
            f"{r['target_val_noisy_mean']:.3e} ± "
            f"{r['target_val_noisy_sd']:.3e}"
        )
        margin_text = (
            f"{r['margin_mean']:.3e} ± "
            f"{r['margin_sd']:.3e}"
        )
        clean_text = (
            f"{r['selected_clean_test_mean']:.3e} ± "
            f"{r['selected_clean_test_sd']:.3e}"
        )
    else:
        val_text = f"{r['target_val_noisy_mean']:.3e}"
        margin_text = f"{r['margin_mean']:.3e}"
        clean_text = f"{r['selected_clean_test_mean']:.3e}"

    paper_rows.append({
        "sigma_noise": f"{r['sigma_noise']:.3f}",
        "Selected correctly": (
            f"{int(r['correct_count'])}/{n}"
        ),
        "Target noisy val. loss": val_text,
        "Margin": margin_text,
        "Selected clean test loss": clean_text,
    })

paper_table_df = pd.DataFrame(paper_rows)
paper_table_df.to_csv(
    PAPER_TABLE_CSV,
    index=False,
)

print("\nManuscript-friendly table:")
display(paper_table_df)

In [ ]:
# ------------------------------------------------------------
# Baseline sanity check at seed=0, sigma=0
# ------------------------------------------------------------

# Values from the original single-seed PAL-uniform baseline.
EXPECTED_AB_VAL = 2.2624128632742213e-06
EXPECTED_B_VAL  = 5.516014e-03

d0 = candidate_df[
    (candidate_df["seed"] == 0)
    & np.isclose(candidate_df["sigma_noise"], 0.0)
]

if len(d0) == len(candidates):
    observed_ab = float(
        d0.loc[
            d0["candidate"] == "AB",
            "val_loss_noisy",
        ].iloc[0]
    )
    observed_b = float(
        d0.loc[
            d0["candidate"] == "B",
            "val_loss_noisy",
        ].iloc[0]
    )

    print("Baseline sanity check (seed=0, sigma=0)")
    print("-----------------------------------------")
    print(
        f"AB: observed={observed_ab:.9e}, "
        f"baseline={EXPECTED_AB_VAL:.9e}"
    )
    print(
        f"B : observed={observed_b:.9e}, "
        f"baseline={EXPECTED_B_VAL:.9e}"
    )
    print(
        "margin observed =",
        f"{observed_b - observed_ab:.9e}",
    )
else:
    print("Baseline sanity check unavailable: condition is incomplete.")

In [ ]:
# ------------------------------------------------------------
# Diagnostic tables: who is the best wrong candidate?
# ------------------------------------------------------------

if len(run_summary_df):
    print("Best wrong candidate by noise level:")
    display(
        pd.crosstab(
            run_summary_df["sigma_noise"],
            run_summary_df["best_wrong_candidate"],
        )
    )

    print("\nPer-run margins:")
    display(
        run_summary_df[
            [
                "seed",
                "sigma_noise",
                "selected_candidate",
                "best_wrong_candidate",
                "validation_margin",
            ]
        ]
    )

In [ ]:
# ------------------------------------------------------------
# Plot 1: validation margin versus observation noise
# ------------------------------------------------------------

if len(summary_df):
    plt.figure(figsize=(7, 5))

    if len(SEEDS) > 1:
        plt.errorbar(
            summary_df["sigma_noise"],
            summary_df["margin_mean"],
            yerr=summary_df["margin_sd"],
            marker="o",
            capsize=4,
        )
    else:
        plt.plot(
            summary_df["sigma_noise"],
            summary_df["margin_mean"],
            marker="o",
        )

    plt.axhline(0, linewidth=1)
    plt.xlabel("Observation-noise standard deviation")
    plt.ylabel("Validation margin")
    plt.title("PAL validation margin versus observation noise")
    plt.grid(True)
    plt.show()

In [ ]:
# ------------------------------------------------------------
# Plot 2: selected clean-test loss versus observation noise
# ------------------------------------------------------------

if len(summary_df):
    plt.figure(figsize=(7, 5))

    if len(SEEDS) > 1:
        plt.errorbar(
            summary_df["sigma_noise"],
            summary_df["selected_clean_test_mean"],
            yerr=summary_df["selected_clean_test_sd"],
            marker="o",
            capsize=4,
        )
    else:
        plt.plot(
            summary_df["sigma_noise"],
            summary_df["selected_clean_test_mean"],
            marker="o",
        )

    plt.yscale("log")
    plt.xlabel("Observation-noise standard deviation")
    plt.ylabel("Selected model: clean test loss")
    plt.title("PAL recovery of the clean target under noisy observations")
    plt.grid(True)
    plt.show()

In [ ]:
# ------------------------------------------------------------
# Package compact results for download
# ------------------------------------------------------------

archive_base = ROOT / RUN_TAG
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Result archive:", archive_path)
print("Files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

## Interpretation guide

The key quantities are:

1. **selection frequency** for `AB`;
2. **validation margin**
   \[
   L_{\rm val}(\text{best wrong})-L_{\rm val}(AB);
   \]
3. the identity of the **best wrong candidate** (especially whether it remains `B`);
4. **clean-target test loss** after training on noisy observations.

A positive validation margin means that `AB` still beats every wrong candidate.
A negative margin means that at least one wrong mapping-class candidate has overtaken it.

The experiment should not be read as a direct numerical test of the finite-sample Hoeffding
proposition: here the PAL models are retrained separately at every noise level, so the trained
predictors and their population risks change with the noise level.